In [4]:
import sys
sys.path.append("..")  # add z2gauge/scripts to path

from hamiltonian.z2ham import *
from hamiltonian.ite import *

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from functools import reduce


In [6]:
Lx, Ly = 2, 3

print(f"Toric code: {Lx}x{Ly} lattice")
print(f"Qubits: {2*Lx*Ly}, Hilbert space dim: {2**(2*Lx*Ly)}\n")

# Debug: Check Wilson loop structure
print("=" * 50)
print("WILSON LOOP STRUCTURE CHECK")
print("=" * 50)
W1 = wilson_loop(Lx, Ly, 0, 0, 1, 1)
W2 = wilson_loop(Lx, Ly, 0, 0, 2, 2)
print(f"1×1 Wilson loop: {jnp.count_nonzero(jnp.diag(W1))} diagonal elements")
print(f"2×2 Wilson loop: {jnp.count_nonzero(jnp.diag(W2))} diagonal elements")
print(f"W1^2 = I? {jnp.allclose(W1 @ W1, jnp.eye(len(W1)))}")
print(f"W2^2 = I? {jnp.allclose(W2 @ W2, jnp.eye(len(W2)))}\n")

# Construct Hamiltonian in deconfined phase (large j_a, small h)
print("=" * 50)
print("DECONFINED PHASE (j_a >> h)")
print("=" * 50)
star_ops = sum_star_operators(Lx, Ly)
trans_ops = transverse_field(Lx, Ly)
H_deconfined = hamiltonian(j_a=-1.0, h=-0.01, star_ops=star_ops, trans_ops=trans_ops)

# Get ground state
eigenvalues, eigenvectors = jnp.linalg.eigh(H_deconfined)
ground_state = eigenvectors[:, 0]
print(f"Ground state energy: {eigenvalues[0]:.4f}")

# Verify phase
phase_info = verify_phase(ground_state, Lx, Ly)
print(f"Phase: {phase_info['phase']}")
print(f"Wilson loops: {phase_info['wilson_loops']}")
print(f"Entanglement entropy: {phase_info['entanglement_entropy']:.4f}")

# Construct Hamiltonian in confined phase (small j_a, large h)
print("\n" + "=" * 50)
print("CONFINED PHASE (h >> j_a)")
print("=" * 50)
H_confined = hamiltonian(j_a=-1.0, h=-1.2, star_ops=star_ops, trans_ops=trans_ops)

# Get ground state
eigenvalues, eigenvectors = jnp.linalg.eigh(H_confined)
ground_state = eigenvectors[:, 0]
print(f"Ground state energy: {eigenvalues[0]:.4f}")

# Verify phase
phase_info = verify_phase(ground_state, Lx, Ly)
print(f"Phase: {phase_info['phase']}")
print(f"Wilson loops: {phase_info['wilson_loops']}")
print(f"Entanglement entropy: {phase_info['entanglement_entropy']:.4f}")

Toric code: 2x3 lattice
Qubits: 12, Hilbert space dim: 4096

WILSON LOOP STRUCTURE CHECK
1×1 Wilson loop: 4096 diagonal elements
2×2 Wilson loop: 4096 diagonal elements
W1^2 = I? True
W2^2 = I? True

DECONFINED PHASE (j_a >> h)
Ground state energy: -6.0005
Phase: confined
Wilson loops: {1: -0.0030239447951316833, 2: 0.9998753070831299}
Entanglement entropy: 1.1139

CONFINED PHASE (h >> j_a)
Ground state energy: -15.0318
Phase: confined
Wilson loops: {1: 9.936513833963545e-08, 2: 0.05739910155534744}
Entanglement entropy: 0.3138


In [9]:
# use ite to get ground state
nsteps = 150

print("=" * 50)
print("DECONFINED PHASE (j_a >> h)")
print("=" * 50)
ground_state, E_f = ite_ground_state(H_deconfined, n_steps=nsteps)
print(f"Ground state energy: {E_f:.4f}")
# verify phase
phase_info = verify_phase(ground_state, Lx, Ly)
print(f"Phase: {phase_info['phase']}")
print(f"Wilson loops: {phase_info['wilson_loops']}")
print(f"Entanglement entropy: {phase_info['entanglement_entropy']:.4f}")

print("=" * 50)
print("CONFINED PHASE (h >> j_a)")
print("=" * 50)
ground_state, E_f = ite_ground_state(H_confined, n_steps=nsteps)
print(f"Ground state energy: {E_f:.4f}")
# verify phase
phase_info = verify_phase(ground_state, Lx, Ly)
print(f"Phase: {phase_info['phase']}")
print(f"Wilson loops: {phase_info['wilson_loops']}")
print(f"Entanglement entropy: {phase_info['entanglement_entropy']:.4f}")


DECONFINED PHASE (j_a >> h)
Ground state energy: -6.0000
Phase: confined
Wilson loops: {1: 0.05669403448700905, 2: 0.9998537302017212}
Entanglement entropy: 2.5593
CONFINED PHASE (h >> j_a)
Ground state energy: -15.0159
Phase: confined
Wilson loops: {1: -9.601457350072451e-06, 2: 0.057593926787376404}
Entanglement entropy: 0.3161
